# CarCrashNet in FiftyOne — a real-data crash-simulation explorer

This notebook builds a [FiftyOne](https://docs.voxel51.com) demo around
**CarCrashNet** (Elrefaie, Shu, Klenk & Ahmed, MIT / Toyota Research Institute,
[arXiv:2605.07098](https://arxiv.org/abs/2605.07098)), the first large-scale
open benchmark for data-driven structural crash simulation — 14,742
bumper-beam pole-impact simulations and 825 full-vehicle crash simulations
(Dodge Neon, Toyota Yaris, Chevrolet Silverado), run in OpenRadioss and
validated against Ansys LS-DYNA and physical crash tests.

**Environment.** This notebook assumes FiftyOne 1.17 running in a Python
virtual environment:

```bash
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install fiftyone==1.17.* fiftyone-brain requests imageio imageio-ffmpeg pandas scikit-learn pillow
jupyter lab
```


## ⚠️ A note on data availability — read this first

As of this writing, the CarCrashNet GitHub repo
(https://github.com/Mohamedelrefaie/CarCrashNet) states explicitly:

> *"The CarCrashNet datasets will be publicly released upon acceptance /
> completion of peer review... We will update this page with download links
> and loading utilities... once the peer-review process is complete."*

In other words: the **raw per-case field data** (the 6.65 TB of VTKHDF mesh
trajectories — displacement, von Mises stress, plastic strain, per node/element,
per timestep, per simulation) is **not yet downloadable**. Nothing here is
synthetic or fabricated to fill that gap — instead, this notebook uses only
what genuinely *is* public right now, and it's more substantial than it might
sound:

1. **Real preview videos/figures** shipped in the repo's `assets/` folder —
   actual renders of actual OpenRadioss simulations from the paper (not
   stock footage, not AI-generated), covering displacement, von Mises stress,
   and plastic strain fields from three synchronized camera angles, plus a
   synced OpenRadioss-vs-LS-DYNA validation clip.
2. **Real numbers** reported in the paper/README: design-space ranges,
   per-vehicle simulation and structural-group counts, solver-validation
   error percentages, and the full CrashSolver-vs-baselines benchmark table
   (RMSE / MAE / relative L2 for Dodge Neon, Toyota Yaris, and Chevrolet
   Silverado).

The last section of this notebook sketches the ingestion code for the raw
VTKHDF field trajectories (mesh → colored point cloud → FiftyOne group
slices per timestep) so that swapping in the full dataset later, once
released, is a drop-in extension rather than a rewrite.


## 1. Setup

In [ ]:
import os
import json
import shutil
import subprocess
from pathlib import Path

import requests
import pandas as pd

import fiftyone as fo
import fiftyone.core.metadata as fom

print("FiftyOne version:", fo.__version__)

DATA_DIR = Path("./carcrashnet_demo_data").resolve()
RAW_DIR = DATA_DIR / "raw"          # downloaded .gif / .png assets
VIDEO_DIR = DATA_DIR / "video"      # .gif -> .mp4 conversions
for d in (RAW_DIR, VIDEO_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Working directory:", DATA_DIR)


## 2. Download the real preview assets

These are fetched straight from the CarCrashNet GitHub repo's `assets/`
folder — the actual figures and GIFs the authors published alongside the
paper.


In [ ]:
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/Mohamedelrefaie/CarCrashNet/main/assets"

# (relative_path_on_github, local_filename)
FIGURES = [
    "figures/CarCrashNet_MainFigure_Overview.png",
    "figures/bumperbeam_mainFigure.png",
    "figures/vehicle_velocity_extremes.png",
    "figures/vehicle_design_space_highlight.png",
    "figures/CrashSolver_mainImage_neurips.png",
    "figures/CarCrashNet_mainFigure.png",
]

# Synchronized 3-camera-angle comparison GIFs across the bumper-beam design
# space, for each of the three released physical fields.
FIELD_VIEW_GIFS = {
    ("displacement", "iso"):  "gifs/comparison_displacement_iso.gif",
    ("displacement", "side"): "gifs/comparison_displacement_side.gif",
    ("displacement", "top"):  "gifs/comparison_displacement_top.gif",
    ("von_mises", "iso"):     "gifs/comparison_vonmises_iso.gif",
    ("von_mises", "side"):    "gifs/comparison_vonmises_side.gif",
    ("von_mises", "top"):     "gifs/comparison_vonmises_top.gif",
    ("plastic_strain", "iso"):  "gifs/comparison_plastic_strain_iso.gif",
    ("plastic_strain", "side"): "gifs/comparison_plastic_strain_side.gif",
    ("plastic_strain", "top"):  "gifs/comparison_plastic_strain_top.gif",
}

SOLVER_VALIDATION_GIF = "gifs/openradioss_vs_lsdyna_synced_direct.gif"


def download(rel_path: str) -> Path:
    url = f"{GITHUB_RAW_BASE}/{rel_path}"
    dest = RAW_DIR / Path(rel_path).name
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    return dest


downloaded_figures = {rel: download(rel) for rel in FIGURES}
downloaded_gifs = {key: download(rel) for key, rel in FIELD_VIEW_GIFS.items()}
validation_gif = download(SOLVER_VALIDATION_GIF)

print(f"Downloaded {len(downloaded_figures)} figures, "
      f"{len(downloaded_gifs)} comparison GIFs, and the solver-validation GIF.")


## 3. Convert GIFs → MP4

FiftyOne's App expects browser-playable video (H.264/mp4), so we transcode
each real GIF once. `imageio` + the bundled `imageio-ffmpeg` binary means no
system `ffmpeg` install is required.


In [ ]:
import importlib
import subprocess
import sys

try:
    import imageio_ffmpeg  # noqa: F401
except ImportError:
    print("Installing imageio-ffmpeg (bundles a static ffmpeg binary, no system ffmpeg needed)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"], check=True)
    importlib.invalidate_caches()

import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageSequence

# Clean slate: previous mp4s from this pipeline were corrupted (broken H.264
# headers -- "Could not load meta information" / "Result too large" when
# reopened). Root cause: individual GIF frames can come back from Pillow at
# slightly different sizes (partial-frame/disposal optimization), and
# letting imageio-ffmpeg auto-pad each frame to a multiple-of-16 canvas
# *independently* produces an inconsistent frame size mid-stream, which
# corrupts the container. Fix: force every frame onto one fixed, pre-padded
# canvas ourselves, then tell the ffmpeg writer not to pad again.
shutil.rmtree(VIDEO_DIR, ignore_errors=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)


def gif_to_mp4(gif_path: Path, fps: int = 12) -> Path:
    mp4_path = VIDEO_DIR / (gif_path.stem + ".mp4")
    if mp4_path.exists() and mp4_path.stat().st_size > 0:
        return mp4_path

    with Image.open(gif_path) as im:
        canvas_size = im.size  # the GIF's logical screen size, fixed for the whole file
        frames = []
        for frame in ImageSequence.Iterator(im):
            rgb = frame.convert("RGB")
            if rgb.size != canvas_size:
                # Some GIFs store partial-frame updates at a smaller size;
                # normalize every frame onto the full canvas.
                rgb = rgb.resize(canvas_size)
            frames.append(np.array(rgb, copy=True))

    # Pad once, to a multiple of 16, identically for every frame -- instead
    # of letting the ffmpeg writer recompute (and potentially vary) padding
    # per frame.
    w, h = canvas_size
    pad_w = (16 - w % 16) % 16
    pad_h = (16 - h % 16) % 16
    if pad_w or pad_h:
        frames = [np.pad(f, ((0, pad_h), (0, pad_w), (0, 0)), mode="edge") for f in frames]

    n_unique = len({f.tobytes() for f in frames})
    print(f"{gif_path.name}: {len(frames)} frames, "
          f"{w}x{h} -> {w + pad_w}x{h + pad_h}, {n_unique} visually distinct")
    if n_unique <= 1:
        print(f"  WARNING: {gif_path.name} has no motion between frames -- "
              f"check the source GIF itself, this isn't a conversion issue.")

    imageio.mimsave(
        str(mp4_path), frames, format="FFMPEG", fps=fps,
        codec="libx264", quality=8, macro_block_size=1,  # we already padded; don't let it pad again
    )

    # Self-check immediately: fail loudly here rather than downstream.
    with imageio.get_reader(str(mp4_path)) as r:
        length = r.get_length()
        if length <= 0:
            raise RuntimeError(f"{mp4_path.name} failed self-check (length={length})")

    return mp4_path


field_view_videos = {key: gif_to_mp4(path) for key, path in downloaded_gifs.items()}
validation_video = gif_to_mp4(validation_gif, fps=15)

print("\nConverted", len(field_view_videos), "comparison videos + 1 validation video.")


## 4. Dataset 1 — synchronized camera-angle crash viewer (FiftyOne groups)

Each of the three released physical fields (displacement, von Mises stress,
equivalent plastic strain) was rendered from three camera angles (isometric,
side, top) of the *same* bumper-beam pole-impact design-space sweep. That's
exactly what FiftyOne's **group slices** are built for — the same pattern
used for multi-camera autonomous-driving rigs, repurposed here for
synchronized simulation viewpoints. Switch fields/views in the App and the
group viewer keeps them in lock-step.


In [ ]:
DATASET_NAME_VIEWS = "carcrashnet-camera-views"
if DATASET_NAME_VIEWS in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME_VIEWS)

views_dataset = fo.Dataset(DATASET_NAME_VIEWS)
views_dataset.add_group_field("group", default="iso")
views_dataset.persistent = True

FIELD_DESCRIPTIONS = {
    "displacement": "Nodal displacement magnitude U(t) relative to the undeformed reference mesh X(0).",
    "von_mises": "Von Mises equivalent stress sigma_vm(t), the standard yield/failure indicator.",
    "plastic_strain": "Equivalent plastic strain (accumulated permanent deformation).",
}

samples = []
fields_seen = sorted({field for field, _ in field_view_videos})
for field in fields_seen:
    group = fo.Group()
    for view in ("iso", "side", "top"):
        video_path = field_view_videos[(field, view)]
        sample = fo.Sample(filepath=str(video_path), group=group.element(view))
        sample["quantity"] = field
        sample["quantity_description"] = FIELD_DESCRIPTIONS[field]
        sample["camera_view"] = view
        sample["campaign"] = "bumper_beam_pole_impact"
        sample["num_simulations_in_campaign"] = 14742
        sample["source"] = "CarCrashNet repo assets/gifs (real OpenRadioss renders)"
        sample.tags.append("real-data")
        samples.append(sample)

views_dataset.add_samples(samples)
views_dataset.compute_metadata()
print(views_dataset)


## 5. Dataset 2 — solver validation clip (OpenRadioss vs. Ansys LS-DYNA)

This is the actual side-by-side, time-synchronized clip the authors used to
validate their open-source OpenRadioss workflow against the commercial
industry-standard solver on the Toyota Yaris model. The quantitative
agreement numbers attached below are the real reported percentages, not
estimates.


In [ ]:
DATASET_NAME_VALIDATION = "carcrashnet-solver-validation"
if DATASET_NAME_VALIDATION in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME_VALIDATION)

validation_dataset = fo.Dataset(DATASET_NAME_VALIDATION)
validation_dataset.persistent = True

vsample = fo.Sample(filepath=str(validation_video))
vsample["comparison"] = "OpenRadioss (open-source) vs Ansys LS-DYNA (commercial)"
vsample["vehicle"] = "Toyota Yaris"
vsample["cfc60_peak_wall_force_pct_diff"] = 7.2
vsample["wall_force_duration_pct_diff"] = 2.6
vsample["peak_internal_energy_pct_diff"] = 0.5
vsample["vs_physical_test_impact_speed_pct_diff"] = 0.2
vsample["vs_physical_test_peak_wall_force_overprediction_pct"] = 14.6
vsample["vs_physical_test_wall_force_duration_underprediction_pct"] = 19.6
vsample["source"] = "CarCrashNet repo assets/gifs (real solver-validation render)"
vsample.tags.append("real-data")

validation_dataset.add_samples([vsample])
validation_dataset.compute_metadata()
print(validation_dataset)


## 6. Dataset 3 — dataset overview figures

The paper's real overview figures, each tagged with the actual reported
dataset-scale numbers so they're filterable/sortable like any other FiftyOne
field rather than just static pictures.


In [ ]:
DATASET_NAME_FIGURES = "carcrashnet-figures"
if DATASET_NAME_FIGURES in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME_FIGURES)

figures_dataset = fo.Dataset(DATASET_NAME_FIGURES)
figures_dataset.persistent = True

FIGURE_METADATA = {
    "CarCrashNet_MainFigure_Overview.png": dict(
        title="CarCrashNet framework overview",
        description="Bumper-beam + three full-vehicle campaigns, ML tasks, and solver validation pipeline.",
        total_simulations=15567, total_size_tb=6.65,
    ),
    "bumperbeam_mainFigure.png": dict(
        title="Bumper-beam pole-impact setup",
        description="DP1000 bumper beam + DP600 crash boxes impacting a rigid cylindrical pole.",
        total_simulations=14742, total_size_tb=None,
    ),
    "vehicle_velocity_extremes.png": dict(
        title="Low- vs high-velocity crash cases across vehicles",
        description="Representative low- and high-speed frontal impacts for Yaris, Neon, and Silverado.",
        total_simulations=825, total_size_tb=None,
    ),
    "vehicle_design_space_highlight.png": dict(
        title="Edited structural groups per vehicle",
        description="Front-support and lower-rail/subframe shell-thickness groups varied +/-10%% per campaign.",
        total_simulations=825, total_size_tb=None,
    ),
    "CrashSolver_mainImage_neurips.png": dict(
        title="CrashSolver architecture",
        description="Hierarchical part-aware neural solver: local component encoders, global transformer, "
                     "interface message passing, temporal nodal readout.",
        total_simulations=None, total_size_tb=None,
    ),
    "CarCrashNet_mainFigure.png": dict(
        title="Full benchmark comparison figure",
        description="CrashSolver vs. Transolver / FIGConvUNet / GeoTransolver across all three vehicle datasets.",
        total_simulations=825, total_size_tb=None,
    ),
}

fig_samples = []
for rel_path, local_path in downloaded_figures.items():
    meta = FIGURE_METADATA[local_path.name]
    s = fo.Sample(filepath=str(local_path))
    for k, v in meta.items():
        if v is not None:
            s[k] = v
    s["source"] = "CarCrashNet repo assets/figures (real published figure)"
    s.tags.append("real-data")
    fig_samples.append(s)

figures_dataset.add_samples(fig_samples)
figures_dataset.compute_metadata()
print(figures_dataset)


## 7. Dataset 4 — CrashSolver benchmark leaderboard

The full, real results table from the README/paper: RMSE, MAE, and relative
L2 (position and displacement) for CrashSolver vs. three state-of-the-art
geometric/transformer baselines, on the held-out test split of each of the
three vehicle datasets. Every sample points at the same benchmark figure but
carries its own real metric fields, so the FiftyOne grid becomes a
sortable/filterable leaderboard.


In [ ]:
DATASET_NAME_BENCH = "carcrashnet-benchmarks"
if DATASET_NAME_BENCH in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME_BENCH)

bench_dataset = fo.Dataset(DATASET_NAME_BENCH)
bench_dataset.persistent = True

# Real numbers, transcribed directly from the CarCrashNet README benchmark table.
BENCHMARK_ROWS = [
    # vehicle, model, rmse_mm, mae_mm, rel_l2_pos, rel_l2_disp
    ("Dodge Neon", "CrashSolver",   32.763, 18.036, 0.02499, 0.08837),
    ("Dodge Neon", "Transolver",    33.947, 18.678, 0.02589, 0.09148),
    ("Dodge Neon", "FIGConvUNet",   34.044, 18.850, 0.02597, 0.09196),
    ("Dodge Neon", "GeoTransolver", 34.403, 18.973, 0.02628, 0.09349),

    ("Toyota Yaris", "CrashSolver",   21.769, 13.507, 0.01537, 0.09043),
    ("Toyota Yaris", "GeoTransolver", 21.773, 13.359, 0.01537, 0.09059),
    ("Toyota Yaris", "FIGConvUNet",   21.910, 13.576, 0.01547, 0.09105),
    ("Toyota Yaris", "Transolver",    22.583, 14.049, 0.01594, 0.09391),

    ("Chevrolet Silverado", "CrashSolver",   61.536, 37.753, 0.03143, 0.17069),
    ("Chevrolet Silverado", "GeoTransolver", 79.230, 45.366, 0.04049, 0.21844),
    ("Chevrolet Silverado", "Transolver",    83.971, 47.510, 0.04291, 0.23184),
    ("Chevrolet Silverado", "FIGConvUNet",  102.747, 62.405, 0.05248, 0.28432),
]

benchmark_figure_path = downloaded_figures["figures/CarCrashNet_mainFigure.png"]

bench_samples = []
for vehicle, model, rmse, mae, rel_l2_pos, rel_l2_disp in BENCHMARK_ROWS:
    s = fo.Sample(filepath=str(benchmark_figure_path))
    s["vehicle"] = vehicle
    s["model"] = model
    s["rmse_mm"] = rmse
    s["mae_mm"] = mae
    s["rel_l2_pos"] = rel_l2_pos
    s["rel_l2_disp"] = rel_l2_disp
    s["source"] = "CarCrashNet README benchmark table (real reported numbers)"
    s.tags.append("real-data")
    bench_samples.append(s)

bench_dataset.add_samples(bench_samples)

# Tag the best model per vehicle (lowest RMSE) so it is easy to filter on in the App.
df = bench_dataset.values(["id", "vehicle", "rmse_mm"])
df = pd.DataFrame({"id": df[0], "vehicle": df[1], "rmse_mm": df[2]})
best_ids = set(df.loc[df.groupby("vehicle")["rmse_mm"].idxmin(), "id"])
for s in bench_dataset:
    if s.id in best_ids:
        s.tags.append("best-in-class")
        s.save()

bench_dataset.compute_metadata()
print(bench_dataset)
bench_dataset.head(3)


## 8. Launch the App

Starts with the synchronized multi-camera crash viewer — the most visually
compelling of the four datasets. Switch to the other three with the dataset
selector in the App, or by re-pointing `session.dataset` below.


In [ ]:
session = fo.launch_app(views_dataset)
# session.dataset = validation_dataset   # OpenRadioss vs LS-DYNA
# session.dataset = figures_dataset      # dataset overview figures
# session.dataset = bench_dataset        # CrashSolver benchmark leaderboard


## 9. A few compelling FiftyOne views (ad hoc)

Everything below runs against real, filterable fields — no placeholder data.


In [ ]:
# a) Only the von-Mises-stress comparison, isometric view
von_mises_iso = views_dataset.match(fo.ViewField("quantity") == "von_mises").match(fo.ViewField("camera_view") == "iso")
print("Von Mises / iso samples:", len(von_mises_iso))

# b) Chevrolet Silverado leaderboard, best RMSE first
silverado_leaderboard = (
    bench_dataset
    .match(fo.ViewField("vehicle") == "Chevrolet Silverado")
    .sort_by("rmse_mm")
)
for s in silverado_leaderboard:
    marker = "  <-- best" if "best-in-class" in s.tags else ""
    print(f"{s.model:>14s}  RMSE={s.rmse_mm:7.3f} mm  MAE={s.mae_mm:7.3f} mm{marker}")

# c) CrashSolver's accuracy advantage tends to grow with structural complexity
crashsolver_only = bench_dataset.match(fo.ViewField("model") == "CrashSolver").sort_by("vehicle")
print("\nCrashSolver RMSE across vehicles:")
for s in crashsolver_only:
    print(f"  {s.vehicle:<22s} RMSE={s.rmse_mm:7.3f} mm")


# Push one of these straight into the live App session
session.view = von_mises_iso


## 10. Saved Views — make the browse/filter demos one click

Turn the ad-hoc `.match()` / `.sort_by()` calls above into **saved views**,
so opening these datasets later — from this notebook, a fresh kernel, or
directly in the App's View bar — gets you straight back to the same
slice without re-deriving it in Python.


In [ ]:
# ---- carcrashnet-camera-views: one saved view per physical quantity, one per camera angle
for quantity in ("displacement", "von_mises", "plastic_strain"):
    view = views_dataset.match(fo.ViewField("quantity") == quantity)
    views_dataset.save_view(f"{quantity}_only", view, overwrite=True)

for camera_view in ("iso", "side", "top"):
    view = views_dataset.match(fo.ViewField("camera_view") == camera_view)
    views_dataset.save_view(f"{camera_view}_angle_only", view, overwrite=True)

# ---- carcrashnet-benchmarks: leaderboards + a best-in-class shortcut
bench_dataset.save_view("best_in_class", bench_dataset.match_tags("best-in-class"), overwrite=True)

for vehicle in ("Dodge Neon", "Toyota Yaris", "Chevrolet Silverado"):
    slug = vehicle.lower().replace(" ", "_")
    view = bench_dataset.match(fo.ViewField("vehicle") == vehicle).sort_by("rmse_mm")
    bench_dataset.save_view(f"{slug}_leaderboard", view, overwrite=True)

bench_dataset.save_view(
    "crashsolver_across_vehicles",
    bench_dataset.match(fo.ViewField("model") == "CrashSolver").sort_by("vehicle"),
    overwrite=True,
)

print("Camera-view saved views:", views_dataset.list_saved_views())
print("Benchmark saved views:  ", bench_dataset.list_saved_views())

# Jump the live App session straight to one of them
session.dataset = views_dataset
session.view = views_dataset.load_saved_view("von_mises_only")


## 11. The real "wow" — an embedding-space explorer for crash frames

Everything so far has been browse-and-filter: useful, but it's still *you*
telling FiftyOne what to look at. This flips that around. We sample frames
from all 10 real crash videos, embed them, and project them into 2D with
FiftyOne Brain — then click around the resulting scatter plot and watch the
matching frames light up in the grid.

What to actually look for once it's up:

- **Do the three physical quantities separate?** `displacement`, `von_mises`,
  and `plastic_strain` are three different renders of the *same* underlying
  simulations. If the embedding clusters mostly by quantity, the color/field
  map dominates the visual signal more than the underlying deformation does.
- **Do camera angles form sub-clusters within each quantity?** `iso` / `side`
  / `top` are literally the same crash from three viewpoints. Distinct
  sub-clusters mean viewpoint dominates; intermixing means the embedding is
  seeing through to shared structure.
- **Color by `frame_time_fraction`.** Frames are ordered snapshots of a
  *continuous* crash, so each source video's frames should trace a connected
  path through the embedding space as the impact progresses (low → high
  fraction) rather than land anywhere at random. That path is the clearest
  single piece of evidence the embedding is capturing something physically
  meaningful, not just noise.

This uses `fiftyone.brain.compute_visualization`, trying real CLIP
embeddings first and falling back to plain color-histogram features (no
internet/torch required) if the zoo model can't be downloaded in this
environment.


In [ ]:
import numpy as np
from PIL import Image, ImageSequence

FRAMES_DIR = DATA_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)


def extract_frames_from_gif(gif_path: Path, n_frames: int = 8):
    """Extract n_frames evenly spaced from a GIF, reading directly from source."""
    try:
        with Image.open(gif_path) as im:
            frames = [
                np.array(frame.convert("RGB"), copy=True)
                for frame in ImageSequence.Iterator(im)
            ]
        
        if not frames:
            print(f"  {gif_path.name}: no frames found")
            return []
        
        total = len(frames)
        # Safely compute frame indices
        if total <= 1:
            idxs = [0]
        else:
            idxs = sorted(set(np.linspace(0, total - 1, num=min(n_frames, total), dtype=int).tolist()))
        
        out = []
        for idx in idxs:
            frac = idx / max(total - 1, 1)
            out_path = FRAMES_DIR / f"{gif_path.stem}_f{idx:04d}.png"
            if not out_path.exists():
                import imageio.v2 as imageio
                imageio.imwrite(str(out_path), frames[idx])
            out.append((out_path, idx, round(frac, 3), total))
        
        return out
    except Exception as e:
        print(f"  {gif_path.name}: ERROR - {e}")
        return []


frame_records = []

# Extract directly from the original GIFs, bypassing the broken mp4 files
for (quantity, camera_view), gif_path in downloaded_gifs.items():
    print(f"Extracting frames from {quantity}/{camera_view} (GIF)...")
    for out_path, idx, frac, total in extract_frames_from_gif(gif_path, n_frames=8):
        frame_records.append(dict(
            path=out_path, quantity=quantity, camera_view=camera_view,
            source_file=gif_path.name, frame_index=idx,
            frame_time_fraction=frac, total_frames=total,
        ))

# Solver validation GIF
if validation_gif.exists():
    print(f"Extracting frames from solver-validation GIF...")
    for out_path, idx, frac, total in extract_frames_from_gif(validation_gif, n_frames=8):
        frame_records.append(dict(
            path=out_path, quantity="solver_validation", camera_view="side_by_side",
            source_file=validation_gif.name, frame_index=idx,
            frame_time_fraction=frac, total_frames=total,
        ))

print(f"\nExtracted {len(frame_records)} frames from GIFs.")

DATASET_NAME_FRAMES = "carcrashnet-frame-embeddings"
if DATASET_NAME_FRAMES in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME_FRAMES)

frames_dataset = fo.Dataset(DATASET_NAME_FRAMES)
frames_dataset.persistent = True

frame_samples = []
for rec in frame_records:
    s = fo.Sample(filepath=str(rec["path"]))
    s["quantity"] = rec["quantity"]
    s["camera_view"] = rec["camera_view"]
    s["source_file"] = rec["source_file"]
    s["frame_index"] = rec["frame_index"]
    s["frame_time_fraction"] = rec["frame_time_fraction"]
    s["total_frames"] = rec["total_frames"]
    s.tags.append("real-data")
    frame_samples.append(s)

frames_dataset.add_samples(frame_samples)
frames_dataset.compute_metadata()
print("\n", frames_dataset)


In [ ]:
import importlib
import subprocess
import sys

import fiftyone.brain as fob


def ensure_pkg(import_name, pip_name=None):
    try:
        importlib.import_module(import_name)
    except ImportError:
        pip_name = pip_name or import_name
        print(f"Installing {pip_name}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
        importlib.invalidate_caches()


def color_histogram_embedding(image_path, bins=16):
    from PIL import Image
    img = Image.open(image_path).convert("RGB").resize((64, 64))
    arr = np.asarray(img)
    hist = np.concatenate([
        np.histogram(arr[..., c], bins=bins, range=(0, 255))[0]
        for c in range(3)
    ]).astype("float32")
    return hist / (hist.sum() + 1e-9)


BRAIN_KEY = "crash_frame_viz"
if BRAIN_KEY in frames_dataset.list_brain_runs():
    frames_dataset.delete_brain_run(BRAIN_KEY)

try:
    print("Attempting CLIP embeddings...")
    results = fob.compute_visualization(
        frames_dataset,
        model="clip-vit-base32-torch",
        brain_key=BRAIN_KEY,
        method="pca",  # PCA works with any embeddings, no numba conflicts
        num_dims=2,
    )
    embedding_method = "CLIP (clip-vit-base32-torch) embeddings + PCA"
except Exception as e:
    print(f"CLIP zoo model unavailable ({type(e).__name__}); "
          f"falling back to color-histogram embeddings + PCA.")
    embeddings = np.stack([color_histogram_embedding(s.filepath) for s in frames_dataset])
    results = fob.compute_visualization(
        frames_dataset,
        embeddings=embeddings,
        brain_key=BRAIN_KEY,
        method="pca",  # PCA: no numba/version conflicts, works everywhere
        num_dims=2,
    )
    embedding_method = "color-histogram features + PCA"

print("Embedding method used:", embedding_method)


In [ ]:
session.dataset = frames_dataset

print("\n" + "="*70)
print("Frame embeddings computed! Your embedding-space explorer is ready.")
print("="*70)
print()
print("To view the 2D scatterplot in the FiftyOne App:")
print()
print("  1. Next to the 'Samples' tab at the top of the grid, click '+'")
print("     to add a new panel")
print("  2. Choose 'Embeddings' from the panel type list")
print("  3. In that panel's dropdown, select the brain key 'crash_frame_viz'")
print("  4. Change the color-by field to 'quantity', 'camera_view', or")
print("     'frame_time_fraction' to re-color the same layout")
print()
print("In the App, you can:")
print("  - Hover over points to see which frame they are")
print("  - Lasso-select a region to highlight matching samples in the grid")
print("  - Filter to a single source_file first, then color by")
print("    frame_time_fraction, to see one crash's progression in isolation")
print()
print("The Embeddings panel can't draw connecting lines between a video's")
print("own frames though -- see the next cell for that.")
print("="*70)


### Actually seeing the "continuous path" claim

The App's Embeddings panel can color points by a field, but it can't draw
connecting lines between frames from the same video -- so on its own it
only gets you partway there.

**Partial, in the App:** filter down to one `source_file` at a time (e.g. in
the Samples grid, filter `source_file == "comparison_von_mises_iso.gif"`),
then color the Embeddings panel by `frame_time_fraction`. With only that
video's 8 points visible, you should see a visible low->high color gradient
across a tight local region rather than random placement.

**Definitive:** reconstruct the trajectory directly from the same 2D
coordinates FiftyOne Brain already computed (`results.points` /
`results.sample_ids`), sort each video's points by `frame_index`, and draw
the connecting line explicitly. That's what the next cell does -- one line
per video, colored by `quantity`, with a hollow square marking each crash's
first frame (t=0) and point fill following `frame_time_fraction`.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd

points = results.points          # (N, 2) array of the same 2D layout used in the App panel
ids = results.sample_ids         # sample id per row, same order as `points`

records = []
for i, sid in enumerate(ids):
    s = frames_dataset[sid]
    records.append(dict(
        x=points[i, 0], y=points[i, 1],
        quantity=s.quantity, camera_view=s.camera_view,
        source_file=s.source_file, frame_index=s.frame_index,
        frame_time_fraction=s.frame_time_fraction,
    ))

df = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(8, 7))
quantities = sorted(df["quantity"].unique())
line_colors = dict(zip(quantities, cm.tab10.colors))

for source_file, group in df.groupby("source_file"):
    group = group.sort_values("frame_index")
    q = group["quantity"].iloc[0]
    c = line_colors[q]

    # the connecting line IS the trajectory through embedding space
    ax.plot(group["x"], group["y"], "-", color=c, alpha=0.5, linewidth=1.5, zorder=2)
    ax.scatter(group["x"], group["y"], c=group["frame_time_fraction"], cmap="viridis",
               s=40, edgecolors=c, linewidths=1.5, zorder=3)
    # hollow square marks t=0 (the undeformed / earliest frame) for that crash
    ax.scatter(group["x"].iloc[0], group["y"].iloc[0], marker="s", s=90,
               facecolors="none", edgecolors=c, linewidths=2, zorder=4)

handles = [plt.Line2D([0], [0], color=c, lw=2, label=q) for q, c in line_colors.items()]
ax.legend(handles=handles, title="quantity (line color)", loc="best")
ax.set_title(
    "Each crash's 8 sampled frames traced through embedding space\n"
    "(hollow square = t=0, point fill = frame_time_fraction via viridis)"
)
ax.set_xlabel("dim 1")
ax.set_ylabel("dim 2")
plt.tight_layout()
plt.show()


## 12. Extending to the full field data (once released)

The raw VTKHDF field trajectories aren't public yet, but the ingestion path
is worth sketching now, since the group-slice pattern above extends
directly from "3 camera angles" to "N timesteps": each simulation case
becomes a **group**, and each timestep becomes a **slice**, letting you
scrub through the actual crash deformation in FiftyOne's 3D visualizer
frame-by-frame, colored by any nodal field (von Mises stress, displacement
magnitude, plastic strain, ...).

This cell is **not executed** — there's no file to point it at yet — but it
shows the real APIs (`h5py`/`vtk` for VTKHDF, `open3d` for point-cloud
export, FiftyOne groups for playback) you'd wire up the moment the dataset
drops.


In [ ]:
# NOT RUN — sketch only, for when CarCrashNet\'s VTKHDF files are released.
#
# import h5py
# import numpy as np
# import open3d as o3d
# import fiftyone as fo
# import matplotlib.cm as cm
#
# def vtkhdf_frame_to_pcd(vtkhdf_path, timestep_index, field="von_mises", out_path=None):
#     """Extract one timestep of one case as a stress/strain-colored point cloud."""
#     with h5py.File(vtkhdf_path, "r") as f:
#         points = f[f"/VTKHDF/Steps/{timestep_index}/Points"][:]        # deformed X(t_n)
#         scalar = f[f"/VTKHDF/Steps/{timestep_index}/PointData/{field}"][:]
#
#     norm = (scalar - scalar.min()) / (np.ptp(scalar) + 1e-9)
#     colors = cm.get_cmap("inferno")(norm)[:, :3]
#
#     pcd = o3d.geometry.PointCloud()
#     pcd.points = o3d.utility.Vector3dVector(points)
#     pcd.colors = o3d.utility.Vector3dVector(colors)
#     out_path = out_path or f"frame_{timestep_index:04d}.pcd"
#     o3d.io.write_point_cloud(out_path, pcd)
#     return out_path
#
# def build_case_group(dataset, case_id, vtkhdf_path, n_timesteps, field="von_mises"):
#     """One FiftyOne group per crash case; one slice per timestep."""
#     group = fo.Group()
#     samples = []
#     for t in range(n_timesteps):
#         pcd_path = vtkhdf_frame_to_pcd(vtkhdf_path, t, field=field)
#         s = fo.Sample(filepath=pcd_path, group=group.element(f"t{t:04d}"))
#         s["case_id"] = case_id
#         s["timestep"] = t
#         s["field"] = field
#         samples.append(s)
#     dataset.add_samples(samples)
